# A9. The lead equations with previous-day inputs, against the lagged-only symbolic search

Question: if every input of Eq. 1 and Eq. 2 is replaced by its previous-day value (no same-day variable), how much skill is left, and does that
beat searching again? Three ways to a "no same-day values" equation, all scored the same way (leave-one-year-out, constants refitted, pooled and
mean-of-years R2, identical days):
  (1) DIRECT: the lead equation with each input shifted by one day, constants refitted.
  (2) LAGGED-ONLY SEARCH (free form): the Pareto front of the lagged-only pipeline (its own feature selection from a pool with no same-day variable),
      every equation re-scored leave-one-year-out (the front values in Table 2 are in-sample).
  (3) CONSTRAINED PySR on the previous-day versions of the lead equations' variables (run_lagged_lead_pysr.py, 3 seeds).
For ozone the lead's inputs shifted by one day are NO, NO2, SW, PM2.5 and east-sector fire (E lag 3 = yesterday's E lag 2).

In [1]:
# Setup
import sys
from pathlib import Path
HERE = Path.cwd().resolve()
sys.path.insert(0, str(HERE.parent)); sys.path.insert(0, str(HERE.parents[1]))
import re, json, warnings
import numpy as np, pandas as pd, sympy as sp
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score
from style import ROOT, DATA_OUT
from common.build_forecast_pool import build_forecast_pool
warnings.filterwarnings("ignore")
B = ROOT / "Data/processed/colombia/bogota"
SAVE = True

In [2]:
# Models: direct previous-day version of each lead equation
def pm25_lead(X, a, b):
    co, u, fn, fe = X; return np.sqrt(np.clip(a * (fe + fn), 0, None)) + b * co / np.sqrt(u)
def o3_lead(X, a, b):
    no, no2, sw, pm, fe = X; return np.sqrt(np.clip(a * sw * np.sqrt(np.clip(pm * no2 / no, 0, None)) + b * fe, 0, None))
SPEC = {"pm25": dict(master="bogota_master_dataset.parquet", model=pm25_lead, p0=[0.104, 15.68],
                     lag_feats=["co_lag1", "wind_speed_era5_lag1", "hotspots_sector_N_lag1", "hotspots_sector_E_lag3"],
                     same_feats=["co", "wind_speed_era5", "hotspots_sector_N", "hotspots_sector_E_lag2"],
                     forecast_json="bogota_forecast_feature_selection.json", forecast_front="bogota_pm25_forecast_interpretable_pysr_pareto.csv"),
        "o3": dict(master="bogota_o3_master_dataset.parquet", model=o3_lead, p0=[0.4916, 0.1079],
                   lag_feats=["no_lag1", "no2_lag1", "radiation_mean_lag1", "pm25_lag1", "hotspots_sector_E_lag3"],
                   same_feats=["no", "no2", "radiation_mean", "pm25", "hotspots_sector_E_lag2"],
                   forecast_json="bogota_o3_forecast_feature_selection.json", forecast_front="bogota_o3_forecast_interpretable_pysr_pareto.csv")}

def loyo(model, X, y, yrs, p0):
    X = tuple(np.asarray(a, float) for a in X)
    try: pf, _ = curve_fit(model, X, y, p0=p0, maxfev=50000)
    except Exception: pf = np.asarray(p0, float)
    pred = np.full(len(y), np.nan)
    for yr in np.unique(yrs):
        te = yrs == yr
        try: p, _ = curve_fit(model, tuple(a[~te] for a in X), y[~te], p0=pf, maxfev=50000)
        except Exception: p = pf
        pred[te] = model(tuple(a[te] for a in X), *p)
    ok = np.isfinite(pred)
    if not ok.all(): return dict(pooled=np.nan, mean_of_years=np.nan)
    per = [r2_score(y[yrs == yr], pred[yrs == yr]) for yr in np.unique(yrs) if (yrs == yr).sum() >= 30]
    return dict(pooled=float(r2_score(y, pred)), mean_of_years=float(np.mean(per)))

FLOAT = re.compile(r"(?<![A-Za-z_0-9.])\d+\.\d+(?:[eE][+-]?\d+)?")
def make_model(eq, feats):
    vals = []
    def sub(m): vals.append(float(m.group(0))); return f"c{len(vals)-1}"
    s = FLOAT.sub(sub, eq)
    syms = {f: sp.Symbol(f) for f in feats}; cs = [sp.Symbol(f"c{i}") for i in range(len(vals))]
    expr = sp.sympify(s, locals={**syms, **{str(c): c for c in cs}, "sqrt": sp.sqrt, "log": sp.log, "exp": sp.exp})
    used = [f for f in feats if syms[f] in expr.free_symbols]; keep = [i for i, c in enumerate(cs) if c in expr.free_symbols]
    fn = sp.lambdify([syms[f] for f in used] + [cs[i] for i in keep], expr, "numpy")
    return (lambda Xt, *p: fn(*Xt, *p)), used, np.array([vals[i] for i in keep])

def score_front(path, data, target, feats, yrs, y):
    fr = pd.read_csv(path); rows = []
    for _, r in fr.iterrows():
        model, used, p0 = make_model(r["equation"], feats)
        if not used: continue
        with np.errstate(all="ignore"): res = loyo(model, tuple(data[f].to_numpy(float) for f in used), y, yrs, p0)
        rows.append(dict(complexity=int(r["complexity"]), equation=r["equation"], **res))
    return pd.DataFrame(rows)

In [3]:
# Run
out_rows, front_tables = [], {}
for pol, s in SPEC.items():
    m = pd.read_parquet(B / s["master"]); m["date"] = pd.to_datetime(m["date"])
    sel = json.load(open(B / s["forecast_json"]))["no_self_lag_no_calendar"]["selected"]
    pool = build_forecast_pool(m, targets=[pol]); pool["year"] = pd.to_datetime(pool["date"]).dt.year
    for c in s["lag_feats"]:
        if c not in pool: pool[c] = m.set_index("date")[c].reindex(pool["date"]).to_numpy() if c in m else np.nan
    for c in s["same_feats"]:                       # the lagged pool carries no same-day columns; take them from the master for the same-day reference
        pool[c] = m.set_index("date")[c].reindex(pd.to_datetime(pool["date"])).to_numpy()
    need = sorted(set(s["lag_feats"]) | set(sel) | set(s["same_feats"]));   # days where the same-day equation can also be evaluated, so every row of the table uses identical days
    d = pool.dropna(subset=need + [pol]).reset_index(drop=True)
    y = d[pol].to_numpy(float); yrs = d.year.to_numpy(); print(pol, "days", len(d))
    # same-day reference on the same rows
    ms = m.copy(); ms["year"] = ms.date.dt.year; ms = ms[ms.date.isin(d.date)].reset_index(drop=True)
    ref = loyo(s["model"], tuple(ms[c].to_numpy(float) for c in s["same_feats"]), ms[pol].to_numpy(float), ms.year.to_numpy(), s["p0"])
    out_rows.append(dict(pollutant=pol, approach="same-day lead equation (reference, same days)", n=len(d), **ref))
    dr = loyo(s["model"], tuple(d[c].to_numpy(float) for c in s["lag_feats"]), y, yrs, s["p0"])
    out_rows.append(dict(pollutant=pol, approach="(1) lead equation, previous-day inputs, constants refitted", n=len(d), **dr))
    ft = score_front(B / s["forecast_front"], d, pol, list(d.columns), yrs, y); ft["pollutant"] = pol
    best = ft.loc[ft.pooled.idxmax()]; at = ft[ft.complexity <= (12 if pol == "pm25" else 15)].sort_values("complexity").iloc[-1]
    out_rows.append(dict(pollutant=pol, approach=f"(2) lagged-only free-form search, front equation at complexity {int(at.complexity)}", n=len(d), pooled=at.pooled, mean_of_years=at.mean_of_years))
    out_rows.append(dict(pollutant=pol, approach=f"(2) lagged-only free-form search, best point (complexity {int(best.complexity)})", n=len(d), pooled=best.pooled, mean_of_years=best.mean_of_years))
    front_tables[pol] = ft
    for seed in (0, 1, 2):
        p = B / f"bogota_{pol}_pysr_lagged_lead_pareto_front_seed{seed}.csv"
        if not p.exists(): continue
        fs = score_front(p, d, pol, list(d.columns), yrs, y); fs["seed"] = seed; front_tables[f"{pol}_c{seed}"] = fs
        for lab, g in (("complexity <= 16", fs[fs.complexity <= 16]), ("any complexity", fs)):
            bb = g.loc[g.pooled.idxmax()]
            out_rows.append(dict(pollutant=pol, approach=f"(3) constrained PySR on previous-day lead variables, seed {seed}, best {lab} (complexity {int(bb.complexity)})", n=len(d), pooled=bb.pooled, mean_of_years=bb.mean_of_years))
res = pd.DataFrame(out_rows); print(res.round(3).to_string(index=False))
if SAVE:
    res.to_csv(DATA_OUT / "lagged_lead_equations.csv", index=False)
    for k, v in front_tables.items(): v.to_csv(DATA_OUT / f"lagged_lead_front_scored_{k}.csv", index=False)

Added fresh lag1/2/3 for 9 columns that didn't already have them: ['wind_dir', 'temp_10m', 'rh_10m', 'blh_min', 'blh_max', 'wind_speed_era5', 'wind_dir_era5', 'wind_dir_sin', 'wind_dir_cos']
pm25 days 3280


Added fresh lag1/2/3 for 10 columns that didn't already have them: ['pm10', 'wind_dir', 'temp_10m', 'rh_10m', 'blh_min', 'blh_max', 'wind_speed_era5', 'wind_dir_era5', 'wind_dir_sin', 'wind_dir_cos']
o3 days 2914


pollutant                                                                                           approach    n  pooled  mean_of_years
     pm25                                                      same-day lead equation (reference, same days) 3280   0.677          0.655
     pm25                                         (1) lead equation, previous-day inputs, constants refitted 3280   0.466          0.414
     pm25                                  (2) lagged-only free-form search, front equation at complexity 12 3280   0.475          0.430
     pm25                                       (2) lagged-only free-form search, best point (complexity 19) 3280   0.480          0.436
     pm25 (3) constrained PySR on previous-day lead variables, seed 0, best complexity <= 16 (complexity 16) 3280   0.489          0.446
     pm25   (3) constrained PySR on previous-day lead variables, seed 0, best any complexity (complexity 24) 3280   0.493          0.448
     pm25 (3) constrained PySR on previou